In [4]:
import pandas as pd
import sys
from jira import JIRA
jira=JIRA('https://research.quantumgalaxy.cn/', basic_auth=('bot2', "jira_bot2"))
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import get_mysql
mysql=get_mysql()

In [5]:
def iter_jira_search(jql,maxResults=500,startAt=0,num_threads=4):
    maxResults=maxResults
    startAt=startAt
    result=[]
    total=jira.search_issues(jql,maxResults=1,startAt=0).total
    while len(result)<=total:
        result1=jira.search_issues(jql,maxResults=maxResults,startAt=startAt)
        
        if result1.current+len(result)>=result1.total:
            print('break')
            print('total issue:',len(result))
            break
        else:
            result+=result1
            print(len(result))
            #print(result1.total)
            #print(len(result)>=result1.total)
            
            startAt+=len(result1)
            #print(startAt)
    return result


In [6]:
import threading
import time
def threading_jira_search(jql, maxResults=50,  num_threads=4):
    start_index=[i*maxResults for i in range(num_threads)]
    result = []
    lock = threading.Lock()
    total = jira.search_issues(jql, maxResults=1, startAt=0).total
    print('total issues:',total)
    
    def search_and_append_issues(startAt):
        nonlocal result
        nonlocal maxResults

        while startAt[0]<total:
            with lock:
                #print('lock once!')
                current_start = startAt[0]
                startAt[0] += maxResults*num_threads
            #print('lock end')
            s_time=time.time()
            result1 = jira.search_issues(jql, maxResults=maxResults, startAt=current_start)
            print('query time:',time.time()-s_time)
            with lock:
                result += result1
            print(current_start, len(result), startAt[0])
            if len(result) >= total:
                print('break this thread')
                print('total issue for now:', len(result))
                break

    thread_list = []

    for i in range(num_threads):
        thread = threading.Thread(target=search_and_append_issues, args=([start_index[i]],))
        thread_list.append(thread)
        thread.start()

    for thread in thread_list:
        thread.join()

    return result

In [15]:
import time

def timeit(func):
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"Time taken by {func.__name__}: {end_time - start_time} seconds")
        return result
    return wrapper
import threading
import queue
@timeit
def worker(jql, maxResults, startAt, endAt, out_queue):
    while startAt < endAt:
        print(startAt)
        actual_maxResults = min(maxResults, endAt - startAt)
        result1 = jira.search_issues(jql, maxResults=actual_maxResults, startAt=startAt)
        #result1 = jira.search_issues(jql, maxResults=maxResults, startAt=startAt)
        for issue in result1:
            out_queue.put(issue)
        startAt += len(result1)
        print(startAt)

def threading_jira_search_2(jql, maxResults=500, startAt=0, num_threads=4):
    result_queue = queue.Queue()
    total = jira.search_issues(jql, maxResults=1, startAt=0).total
    print(total)
    threads = []
    for i in range(num_threads):
        worker_startAt = startAt + i * (total // num_threads)
        worker_endAt = startAt + (i + 1) * (total // num_threads) if i != num_threads - 1 else total
        thread = threading.Thread(target=worker, args=(jql, maxResults, worker_startAt, worker_endAt, result_queue))
        thread.start()
        threads.append(thread)

    for thread in threads:
        thread.join()

    result = []
    while not result_queue.empty():
        result.append(result_queue.get())

    print('total issue:', len(result))
    return result

In [16]:
issues=threading_jira_search_2("project = COMPSTUDY AND updated >=   startOfDay('-1d')")

199
0
49
98
147
49
Time taken by worker: 16.482397079467773 seconds
98
Time taken by worker: 17.919947147369385 seconds
147
Time taken by worker: 21.08199954032898 seconds
199
Time taken by worker: 37.38279843330383 seconds
total issue: 199


In [12]:
total

NameError: name 'total' is not defined

In [9]:
len(issues)

502

In [32]:
issues=threading_jira_search('project = COMPSTUDY AND issuetype = 公司研究',maxResults=500)

total issues: 6436
query time: 171.0008373260498
0 500 2000
query time: 191.11022925376892
500 1000 2500
query time: 213.11709070205688
1000 1500 3000
query time: 215.18345403671265
1500 2000 3500
query time: 179.60752940177917
2500 2500 4500
query time: 208.97277164459229
2000 3000 4000
query time: 190.1310043334961
3500 3500 5500
query time: 199.42305779457092
3000 4000 5000
query time: 263.46990633010864
4500 4500 6500
query time: 255.6300492286682
4000 5000 6000
query time: 499.4172260761261
5000 5500 7000
query time: 781.4130673408508
5500 6000 7500
query time: 788.4258012771606
6000 6436 8000
break this thread
total issue for now: 6436


In [50]:
issue_codes=[issue.fields.customfield_10201 for issue in issues]
len(issue_codes)


6436

In [37]:
stock=mysql.read_query('select code,name from ticker_info where region="CN"')
stock[0]

('000001.SZ', '平安银行')

In [65]:
new_issues=[]
new_names=[]
for code,name in stock:
    if '+' in code:continue
    if code not in issue_codes:
        #print(code,name)
        new_issues.append((code,name))
        pass
    else:
        issue=issues[issue_codes.index(code)]
        if issue.fields.summary !=name:
            #print(issue.fields.summary,name)
            new_names.append((code,name,issue))
            

In [68]:
new_names

[('000020.SZ', '深华发A', <JIRA Issue: key='COMPSTUDY-6832', id='30469'>),
 ('000050.SZ', '深天马A', <JIRA Issue: key='COMPSTUDY-4359', id='18996'>),
 ('000100.SZ', 'TCL科技', <JIRA Issue: key='COMPSTUDY-829', id='11105'>),
 ('000408.SZ', '藏格矿业', <JIRA Issue: key='COMPSTUDY-2123', id='15697'>),
 ('000570.SZ', '苏常柴A', <JIRA Issue: key='COMPSTUDY-6841', id='30487'>),
 ('000620.SZ', '*ST新联', <JIRA Issue: key='COMPSTUDY-5599', id='27578'>),
 ('000629.SZ', '钒钛股份', <JIRA Issue: key='COMPSTUDY-2303', id='16187'>),
 ('000637.SZ', 'ST实华', <JIRA Issue: key='COMPSTUDY-6245', id='29205'>),
 ('000667.SZ', 'ST美置(退市)', <JIRA Issue: key='COMPSTUDY-7048', id='30902'>),
 ('000791.SZ', '甘肃能源', <JIRA Issue: key='COMPSTUDY-2390', id='16403'>),
 ('000818.SZ', '航锦科技', <JIRA Issue: key='COMPSTUDY-5236', id='24458'>),
 ('002006.SZ', '精工科技', <JIRA Issue: key='COMPSTUDY-2593', id='16955'>),
 ('002105.SZ', '信隆健康', <JIRA Issue: key='COMPSTUDY-2391', id='16405'>),
 ('002113.SZ', '*ST天润', <JIRA Issue: key='COMPSTUDY-6479', 

In [69]:
new_issues

[('301261.SZ', '恒工精密'),
 ('301329.SZ', '信音电子'),
 ('301370.SZ', '国科恒泰'),
 ('301381.SZ', '赛维时代'),
 ('301393.SZ', '昊帆生物'),
 ('301456.SZ', '盘古智能'),
 ('301499.SZ', '维科精密'),
 ('301503.SZ', '智迪科技'),
 ('301512.SZ', '智信精密'),
 ('688563.SH', '航材股份'),
 ('688602.SH', '康鹏科技'),
 ('688610.SH', '埃科光电'),
 ('688627.SH', '精智达'),
 ('688638.SH', '誉辰智能'),
 ('832982.BJ', '锦波生物'),
 ('833751.BJ', '惠同新材'),
 ('837592.BJ', '华信永道'),
 ('838701.BJ', '豪声电子'),
 ('873576.BJ', '天力复合'),
 ('U', 'S'),
 ('UR.CZC', 'CZCE 尿素（主连）'),
 ('USDCNY.EX', '美元中间价'),
 ('V.DCE', 'DCE PVC（主连）'),
 ('WH.CZC', 'CZCE强麦'),
 ('WR.SHF', 'SHFE 线材（主连）'),
 ('Y.DCE', 'DCE 豆油（主连）'),
 ('ZC.CZC', 'CZCE 动力煤（主连）'),
 ('ZN.SHF', 'SHFE 锌（主连）')]

In [63]:
for code,name in new_issues:
    try:
        issue=jira.create_issue(project='10000',issuetype='公司研究',summary=name,customfield_10201=code)
    except Exception as e:
        print(e)
        print(code,name)


JiraError HTTP 400 url: https://research.quantumgalaxy.cn/rest/api/2/issue
	text: 万得代码填写错误 或 系统已有重复的万得代码问题，无法创建
	
	response headers = {'X-AREQUESTID': '24x493204x1', 'X-ASESSIONID': '1f1lye3', 'Referrer-Policy': 'strict-origin-when-cross-origin', 'X-XSS-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'X-Frame-Options': 'SAMEORIGIN', 'Content-Security-Policy': "frame-ancestors 'self'", 'Strict-Transport-Security': 'max-age=31536000', 'X-Seraph-LoginReason': 'OK', 'X-AUSERNAME': 'bot2', 'Cache-Control': 'no-cache, no-store, no-transform', 'Content-Encoding': 'gzip', 'Vary': 'User-Agent', 'Content-Type': 'application/json;charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Date': 'Wed, 26 Jul 2023 16:24:07 GMT', 'Connection': 'close'}
	response text = {"errorMessages":[],"errors":{"customfield_10201":"万得代码填写错误 或 系统已有重复的万得代码问题，无法创建"}}
301261.SZ 恒工精密
JiraError HTTP 400 url: https://research.quantumgalaxy.cn/rest/api/2/issue
	text: 万得代码填写错误 或 系统已有重复的万得代码问题，无法创建
	
	response he

In [67]:
for code,name,issue in new_names:
    try:
        issue.update(summary=name)
    except Exception as e:
        print(e)
        print(code,name)


In [38]:
len(stock)

5462

In [4]:
stock[5357:]

(('872541.BJ', '铁大科技'),
 ('872808.BJ', '曙光数创'),
 ('872895.BJ', '花溪科技'),
 ('872925.BJ', '锦好医疗'),
 ('873001.BJ', '纬达光电'),
 ('873122.BJ', '中纺标'),
 ('873152.BJ', '天宏锂电'),
 ('873167.BJ', '新赣江'),
 ('873169.BJ', '七丰精工'),
 ('873223.BJ', '荣亿精密'),
 ('873305.BJ', '九菱科技'),
 ('873339.BJ', '恒太照明'),
 ('873527.BJ', '夜光明'),
 ('873576.BJ', '天力复合'),
 ('873593.BJ', '鼎智科技'),
 ('900929.SH', '锦旅B股'),
 ('900939.SH', '汇丽B'),
 ('900948.SH', '伊泰B股'),
 ('900953.SH', '凯马B'),
 ('900957.SH', '凌云B股'),
 ('A.DCE', 'DCE 黄大豆一号【食用】（主连）'),
 ('AF-S.CFE', 'CFFEX澳元兑美元(AUD/USD)期货(仿真)'),
 ('AG.SHF', 'SHFE 银（主连）'),
 ('Ag(T+D).SGE', 'SGE白银T+D'),
 ('AL.SHF', 'SHFE 铝（主连）'),
 ('AP.CZC', 'CZCE 苹果（主连）'),
 ('AU.SHF', 'SHFE 金（主连）'),
 ('Au(T+D).SGE', 'SGE黄金T+D'),
 ('B.DCE', 'DCE 黄大豆二号【榨油用】（主连）'),
 ('BB.DCE', 'DCE胶合板'),
 ('BC.INE', 'INE国际铜'),
 ('BU.SHF', 'SHFE 沥青（主连）'),
 ('C.DCE', 'DCE 玉米（主连）'),
 ('CF.CZC', 'CZCE 棉花（主连）'),
 ('CJ.CZC', 'CZCE 红枣（主连）'),
 ('CS.DCE', 'DCE 玉米淀粉（主连）'),
 ('CU.SHF', 'SHFE 铜（主连）'),
 ('CY.CZC', 'CZCE棉纱'),
 ('EB.DCE'

In [5]:
for code,name in stock[5357:]:
    if '+' in code:
        print(code.replace('+','\\u002b'))
        print(code)

Ag(T\u002bD).SGE
Ag(T+D).SGE
Au(T\u002bD).SGE
Au(T+D).SGE


In [9]:
'\u002b'

'+'

In [5]:
from tqdm import tqdm

In [18]:

with tqdm(stock[5357:],desc='here') as t:
    for code,name in t:
        if'+' in code:
                
            continue
        get_issue=jira.search_issues(f'project = COMPSTUDY AND 万得代码 ={code}')
        if get_issue:
            pass
            t.set_postfix({'result':'exist issue:'+name})
        else:
            
            
            issue=jira.create_issue(project='10000',issuetype='公司研究',summary=name,customfield_10201=code)
            

            #print('noissue:',name)

            t.set_postfix({'result':'noissue:'+name})
       


here:  89%|████████▉ | 74/83 [04:11<00:30,  3.40s/it, result=noissue:CFFEX3年期国债期货(仿真)]            


JIRAError: JiraError HTTP 400 url: https://research.quantumgalaxy.cn/rest/api/2/issue
	text: 万得代码填写错误 或 系统已有重复的万得代码问题，无法创建
	
	response headers = {'X-AREQUESTID': '676x412931x1', 'X-ASESSIONID': '1q4ddbn', 'Referrer-Policy': 'strict-origin-when-cross-origin', 'X-XSS-Protection': '1; mode=block', 'X-Content-Type-Options': 'nosniff', 'X-Frame-Options': 'SAMEORIGIN', 'Content-Security-Policy': "frame-ancestors 'self'", 'Strict-Transport-Security': 'max-age=31536000', 'X-Seraph-LoginReason': 'OK', 'X-AUSERNAME': 'bot2', 'Cache-Control': 'no-cache, no-store, no-transform', 'Content-Encoding': 'gzip', 'Vary': 'User-Agent', 'Content-Type': 'application/json;charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Date': 'Fri, 14 Jul 2023 03:16:47 GMT', 'Connection': 'close'}
	response text = {"errorMessages":[],"errors":{"customfield_10201":"万得代码填写错误 或 系统已有重复的万得代码问题，无法创建"}}